In [161]:
import os, time, shutil

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import humanize

import typing, collections

import typing

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import joblib
pio.templates.default = "plotly_white"

from sklearn.preprocessing import StandardScaler
from tqdm.autonotebook import tqdm

from sklearn.metrics import mean_absolute_error, mean_squared_error

from IPython.display import display, HTML
# %load_ext autoreload
# %autoreload 2
np.random.seed(42)
tqdm.pandas()

Read the data from previous baseline

In [165]:
train_df = pd.read_parquet("df_train_feature_eng_2.parquet")
validation_df = pd.read_parquet("df_val_feature_eng_2.parquet")
test_df = pd.read_parquet("df_test_feature_eng_2.parquet")

In [166]:
sel_lclids = train_df.LCLid.unique().tolist()
target = 'energy_consumption'
index_cols = ['LCLid', 'timestamp']
train_df.set_index(index_cols, inplace=True, drop=False)
validation_df.set_index(index_cols, inplace=True, drop=False)
test_df.set_index(index_cols, inplace=True, drop=False)

pred_df = pd.concat([train_df[target], test_df[target]])

In [167]:
pred_df[:2]

LCLid      timestamp          
MAC000026  2012-01-01 00:00:00    0.085
           2012-01-01 00:30:00    0.108
Name: energy_consumption, dtype: float64

Handling missing values

In [176]:
''' Null check '''
nc = train_df.isnull().sum()
nc[nc>0].head()

,0
energy_consumption,140797
pressure,3308
energy_consumption_lag_1,140944
energy_consumption_lag_2,141091
energy_consumption_lag_3,141238


In [48]:
missing_value_config = MissingValueConfig(
	bfill_columns=["energy_consumption_lag_1", "energy_consumption_lag_2", "energy_consumption_lag_3", "energy_consumption_lag_4", "energy_consumption_lag_5", "energy_consumption_lag_46","energy_consumption_lag_47", "energy_consumption_lag_48","energy_consumption_lag_49", "energy_consumption_lag_50","energy_consumption_lag_334", "energy_consumption_lag_335",
	"energy_consumption_lag_336", "energy_consumption_lag_337",
	"energy_consumption_lag_338", "energy_consumption_rolling_3_mean",
	"energy_consumption_rolling_3_std", "energy_consumption_rolling_6_mean",
	"energy_consumption_rolling_6_std", "energy_consumption_rolling_12_mean",
	"energy_consumption_rolling_12_std", "energy_consumption_rolling_48_mean",
	"energy_consumption_rolling_48_std", "energy_consumption_48_seasonal_rolling_3_mean","energy_consumption_48_seasonal_rolling_3_std","energy_consumption_336_seasonal_rolling_3_mean",
	"energy_consumption_336_seasonal_rolling_3_std","energy_consumption_ewma__span_2880", "energy_consumption_ewma__span_336",
	"energy_consumption_ewma__span_48",
	],
	ffill_columns=[],
	zero_fill_columns=[],
)

In [ ]:
train_df = missing_value_config.impute_missing_values(train_df)
validation_df = missing_value_config.impute_missing_values(validation_df)
test_df = missing_value_config.impute_missing_values(test_df)

In [50]:
''' Null check again '''
nc = train_df.isnull().sum()
nc[nc>0]

,0


Load previous Forecasting data and compute aggregate metrics on selected LCLids.

In [52]:
ml_forecast_df = pd.read_pickle('ml_single_step_predictions_val_df.pkl')

In [174]:
ml_forecast_df.head()

,predictions,LCLid,Algorithm,energy_consumption
timestamp,,,,
2014-02-01 00:00:00,0.152389,MAC000026,Lasso Regression,0.178
2014-02-01 00:30:00,0.139119,MAC000026,Lasso Regression,0.092
2014-02-01 01:00:00,0.088697,MAC000026,Lasso Regression,0.115
2014-02-01 01:30:00,0.105112,MAC000026,Lasso Regression,0.135
2014-02-01 02:00:00,0.123536,MAC000026,Lasso Regression,0.143


In [118]:
# Corrected evaluate_forecast function
def evaluate_forecast(y_pred, test_target, train_target, model_name):
    # Ensure y_pred and test_target have a MultiIndex with LCLid as the first level
    if not isinstance(y_pred.index, pd.MultiIndex) or y_pred.index.names[0] != 'LCLid':
        pass # Assuming data preparation outside will ensure this
    if not isinstance(test_target.index, pd.MultiIndex) or test_target.index.names[0] != 'LCLid':
        pass # Assuming data preparation outside will ensure this

    metric_l = []
    # Corrected iteration over unique LCLids from the index
    # Using .unique() as remove_unused_categories() is not available on Index/DatetimeIndex
    for _id in tqdm(test_target.index.get_level_values(0).unique(), desc="Calculating metrics..."):
        target = test_target.xs(_id)
        _y_pred = y_pred.xs(_id)

        mae = mean_absolute_error(target, _y_pred)
        rmse = np.sqrt(mean_squared_error(target, _y_pred))

        # Calculate MAPE, handling division by zero for actual values of 0
        mape = np.mean(np.abs((target - _y_pred) / target)) * 100
        mape = mape if np.isfinite(mape) else np.nan

        metric_l.append({
            'LCLid': _id,
            'Model': model_name,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape
        })

    metrics_df = pd.DataFrame(metric_l)
    agg_metrics = metrics_df.drop(columns=['LCLid']).groupby('Model').agg(['mean', 'std'])

    # Flatten the multi-level columns
    agg_metrics.columns = ['_'.join(col).strip() for col in agg_metrics.columns.values]
    agg_metrics = agg_metrics.reset_index()

    return agg_metrics, metrics_df

# --- Data Preparation and Function Call ---
# Create a filtered and re-indexed version of ml_forecast_df for this evaluation
# This ensures LCLid is the first level of the MultiIndex, as expected by evaluate_forecast.
lightgbm_filtered_df = ml_forecast_df[ml_forecast_df.Algorithm =='LightGBM'].copy()

# Reset the index to turn 'timestamp' into a column before setting a new MultiIndex
lightgbm_filtered_df.reset_index(inplace=True)
lightgbm_filtered_df.set_index(['LCLid', 'timestamp'], inplace=True)

# Now, extract the y_pred and test_target Series with the correct MultiIndex
y_pred_lightgbm = lightgbm_filtered_df['predictions']
test_target_lightgbm = lightgbm_filtered_df['energy_consumption']

baseline_agg_metrics, baseline_metrics_df = evaluate_forecast(
    y_pred=y_pred_lightgbm,
    test_target=test_target_lightgbm,
    train_target = train_df['energy_consumption'], # Assuming train_df is already MultiIndexed
    model_name='LightGBM'
)

Calculating metrics...:   0%|          | 0/146 [00:00<?, ?it/s]

In [55]:
baseline_agg_metrics

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,MAPE_mean,MAPE_std
0,LightGBM,0.09522,0.063441,0.161857,0.097073,56.128138,35.21914


In [173]:
baseline_metrics_df.head()

,LCLid,Model,MAE,RMSE,MAPE
0,MAC000026,LightGBM,0.114287,0.194724,29.286626
1,MAC000052,LightGBM,0.102284,0.175501,122.948470
2,MAC000079,LightGBM,0.279891,0.376616,83.221172
3,MAC000085,LightGBM,0.152073,0.240148,55.644659
4,MAC000105,LightGBM,0.090544,0.094021,47.822461


In [57]:
metric_record = [baseline_agg_metrics]

In [58]:
metric_record

[      Model  MAE_mean   MAE_std  RMSE_mean  RMSE_std  MAPE_mean  MAPE_std
 0  LightGBM   0.09522  0.063441   0.161857  0.097073  56.128138  35.21914]

#### PyTorch Tabular

In [59]:
!pip install pytorch_tabular > /dev/null

In [60]:
from pytorch_tabular import TabularModel
from pytorch_tabular.models import BaseModel, FTTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig, ExperimentConfig
from pytorch_tabular.models.common.heads import LinearHeadConfig

In [69]:
data_config = DataConfig(
	target=[target],
	continuous_cols=[
	"visibility", "windBearing", "temperature", "dewPoint", "pressure", "apparentTemperature", "windSpeed", "humidity", "energy_consumption_lag_1", "energy_consumption_lag_2", "energy_consumption_lag_3", "energy_consumption_lag_4", "energy_consumption_lag_5", "energy_consumption_lag_46", "energy_consumption_lag_47", "energy_consumption_lag_48", "energy_consumption_lag_49", "energy_consumption_lag_50", "energy_consumption_lag_334", "energy_consumption_lag_335", "energy_consumption_lag_336", "energy_consumption_lag_337", "energy_consumption_lag_338", "energy_consumption_rolling_3_mean", "energy_consumption_rolling_3_std", "energy_consumption_rolling_6_mean", "energy_consumption_rolling_6_std", "energy_consumption_rolling_12_mean", "energy_consumption_rolling_12_std", "energy_consumption_rolling_48_mean", "energy_consumption_rolling_48_std", "energy_consumption_48_seasonal_rolling_3_mean", "energy_consumption_48_seasonal_rolling_3_std", "energy_consumption_336_seasonal_rolling_3_mean", "energy_consumption_336_seasonal_rolling_3_std", "energy_consumption_ewma_span_2880", "energy_consumption_ewma_span_336", "energy_consumption_ewma_span_48", "timestamp_Elapsed", "timestamp_Month_sin_1", "timestamp_Month_sin_2", "timestamp_Month_sin_3", "timestamp_Month_sin_4", "timestamp_Month_sin_5", "timestamp_Month_cos_1", "timestamp_Month_cos_2", "timestamp_Month_cos_3", "timestamp_Month_cos_4", "timestamp_Month_cos_5", "timestamp_Hour_sin_1", "timestamp_Hour_sin_2", "timestamp_Hour_sin_3", "timestamp_Hour_sin_4", "timestamp_Hour_sin_5", "timestamp_Hour_cos_1", "timestamp_Hour_cos_2", "timestamp_Hour_cos_3", "timestamp_Hour_cos_4", "timestamp_Hour_cos_5", "timestamp_Minute_sin_1", "timestamp_Minute_sin_2", "timestamp_Minute_sin_3", "timestamp_Minute_sin_4", "timestamp_Minute_sin_5", "timestamp_Minute_cos_1", "timestamp_Minute_cos_2", "timestamp_Minute_cos_3", "timestamp_Minute_cos_4", "timestamp_Minute_cos_5", "timestamp_Is_quarter_end", "timestamp_Is_quarter_start", "timestamp_Is_year_end", "timestamp_Is_year_start", "timestamp_Is_month_start",
	],
	categorical_cols=existing_categorical_cols,
	normalize_continuous_features=True
)

In [149]:
trainer_config = TrainerConfig(
    auto_lr_find=False, # Runs the LRFinder to automatically derive a learning rate
    batch_size=1024,
    max_epochs=10,
    #gpus=-1,
)

optimizer_config = OptimizerConfig()

In [150]:
existing_categorical_cols = [col for col in data_config.categorical_cols if col in train_df.columns]

train_df[existing_categorical_cols] = train_df[existing_categorical_cols].astype(str)
validation_df[existing_categorical_cols] = validation_df[existing_categorical_cols].astype(str)
test_df[existing_categorical_cols] = test_df[existing_categorical_cols].astype(str)

#### FT Transformer Model

In [151]:
linear_head_config = LinearHeadConfig(
    layers = "32",
    activation = "ReLU"
)

model_config = FTTransformerConfig(
    task = 'regression',
    num_attn_blocks=3,
    num_heads=4,
    transformer_head_dim=64,
    attn_dropout=0.2,
    ff_dropout=0.1,
    learning_rate=0.001,
    head_config = linear_head_config.__dict__,
    metrics=['mean_squared_error']
)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

2026-09-29 08:22:14,757 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [106]:
from omegaconf import DictConfig, ListConfig
from omegaconf.base import ContainerMetadata, Metadata
from omegaconf.nodes import AnyNode

torch.serialization.add_safe_globals([DictConfig, ContainerMetadata, Metadata, ListConfig, AnyNode, typing.Any, dict, collections.defaultdict, list, int])
tabular_model.fit(train=train_df)

Epoch 0/0  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3561/3561 2:59:55 • 0:00:00 0.34it/s v_num: 15.000 train_loss: 0.032   
                                                                                 valid_loss: 0.034                 
                                                                                 valid_mean_squared_error: 0.034   
                                                                                 train_mean_squared_error: 0.039   

INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


2026-09-29 03:39:20,884 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-09-29 03:39:20,885 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [110]:
tabular_model.save_model('ft_transformer_global')

2026-09-29 07:42:22,451 - {pytorch_tabular.tabular_model:1577} - WARNING - Directory is not empty. Overwriting the contents.
INFO:pytorch_lightning.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.


In [ ]:
shutil.rmtree("saved_models")

In [115]:
forecast_df = tabular_model.predict(test_df, include_input_features=False)
pred_df = pred_df.join(forecast_df[['energy_consumption_prediction']]).rename(columns={'energy_consumption_prediction': model_config._model_name})

In [120]:
agg_metrics, eval_metrics_df = evaluate_forecast(
    y_pred=forecast_df[f'{target}_prediction'],
    test_target=test_df[target],
    train_target = train_df['energy_consumption'], # Assuming train_df is already MultiIndexed
    model_name=model_config._model_name
)

Calculating metrics...:   0%|          | 0/159 [00:00<?, ?it/s]

In [121]:
metric_record.append(agg_metrics)

#### Evaluation of ML Forecast

In [125]:
agg_metrics_df = pd.concat(metric_record)
agg_metrics_df

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,MAPE_mean,MAPE_std
0,LightGBM,0.095220,0.063441,0.161857,0.097073,56.128138,35.219140
0,FTTransformerModel,0.102823,0.071361,0.170918,0.100792,49.954105,28.519559


In [132]:
agg_metrics_df.reset_index(drop=True, inplace=True)
agg_metrics_df.style.format(
    {'MAE_mean': '{:.2f}', 'RMSE_mean': '{:.2f}', 'MAPE_mean': '{:.2f}'}).highlight_min(color='lightgreen', subset=["MAE_mean","RMSE_mean","MAPE_mean"]).apply(highlight_abs_min, subset=["MAE_mean","RMSE_mean","MAPE_mean"])

,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,MAPE_mean,MAPE_std
0,LightGBM,0.10,0.063441,0.16,0.097073,56.13,35.219140
1,FTTransformerModel,0.10,0.071361,0.17,0.100792,49.95,28.519559


In [172]:
metrics_df = pd.concat([eval_metrics_df, baseline_metrics_df])
metrics_df.head()

,LCLid,Model,MAE,RMSE,MAPE
0,MAC000026,FTTransformerModel,0.119052,0.204097,30.642181
1,MAC000052,FTTransformerModel,0.096269,0.176597,91.752567
2,MAC000079,FTTransformerModel,0.330516,0.427772,100.102755
3,MAC000085,FTTransformerModel,0.128189,0.229188,39.304503
4,MAC000105,FTTransformerModel,0.080278,0.090175,33.987878


In [142]:
fig = px.histogram(metrics_df, x = 'MAE', color='Model', pattern_shape='Model', marginal='box', nbins=500, barmode='overlay', histnorm='probability density')

fig = format_plot(fig, xlabel='MAE', ylabel='Probability Density', title='Distribution of MAE in the dataset')

#fig.update_layout(xaxis_range=[0, 0.2])
fig.show()

In [144]:
fig = px.histogram(metrics_df, x = 'RMSE', color='Model', pattern_shape='Model', marginal='box', nbins=500, barmode='overlay', histnorm='probability density')

fig = format_plot(fig, xlabel='RMSE', ylabel='Probability Density', title='Distribution of RMSE in the dataset')

#fig.update_layout(xaxis_range=[0.2])
fig.show()

In [145]:
fig = px.histogram(metrics_df, x = 'MAPE', color='Model', pattern_shape='Model', marginal='box', nbins=500, barmode='overlay', histnorm='probability density')

fig = format_plot(fig, xlabel='MAPE', ylabel='Probability Density', title='Distribution of MAPE in the dataset')

#fig.update_layout(xaxis_range=[0.2])
fig.show()

In [148]:
pred_df.to_pickle('ft_transformer_val_df.pkl')
metrics_df.to_pickle('ft_transformer_metrics_val_df.pkl')
metrics_df.to_pickle('ft_transformer_aggregate_metrics_val.pkl')